In [ ]:
import os

RAP_PROJECT_ID = os.environ["DNANEXUS_PROJECT_ID"]  # set your own DNAnexus RAP project ID
RAP_DATA_ROOT = os.environ["DNANEXUS_DATA_ROOT"]  # complete RAP data root


In [ ]:
import glob
import numpy as np
import polars as pl

In [ ]:
# Get gene trait associations
!dx download {RAP_DATA_ROOT}/REGENIE_results/regenie_127phenotypes_lofteeHC_mac20_EUR.parquet -o PATH_TO_FILE
# !dx download {RAP_DATA_ROOT}/REGENIE_results/loftee_mac20_associations_bh_corrected.parquet -o PATH_TO_FILE

gene_trait_df = (
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)


gene_trait_df

In [ ]:
# EUR unrelated individuals

!dx download {RAP_DATA_ROOT}/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o PATH_TO_FILE

unrel_eur_samples = pl.read_csv('PATH_TO_FILE')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

In [ ]:
# Download phenotypes: covariates and PRS corrected
!dx download {RAP_DATA_ROOT}/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o PATH_TO_FILE

phenos = (
    pl.read_parquet('PATH_TO_FILE')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)


In [ ]:
# Get null counts and transpose for easier viewing
null_counts = phenos.null_count()

# Convert to a format that's easier to read (column name -> count)
null_counts_long = null_counts.transpose(include_header=True, column_names=['null_count'])

less_miss = null_counts_long.filter(pl.col('null_count') <= 0.2*len(unrel_eur_samples)).sort('null_count', descending=True)

traits2keep = gene_trait_df.filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))['phenotype'].unique().to_list()

print(len(traits2keep))

gene_trait_df.filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))

In [ ]:
(
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))
    .write_parquet('PATH_TO_FILE')
)

!dx upload PATH_TO_FILE --path {RAP_DATA_ROOT}/REGENIE_results/
#1_683_399